In [1]:
import sys
import os
sys.path.append(os.path.abspath('..'))

In [ ]:
import json
import pandas as pd
from datasets import load_dataset, DatasetDict
import chromadb
from rag_bench import evaluator, helper

from pipeline.ingestion.chroma import insert_data
from pipeline.ingestion.splitters import SPLITTER_NAMES
from pipeline.retriever import ChromaRetriever

### Загрузка датасетов

In [3]:
CACHE_DIR = "../datasets/"

HIST_PRIVATE_QA_REPO_ID: str = "ai-forever/hist-rag-bench-private-qa"
HIST_PRIVATE_TEXTS_REPO_ID: str = "ai-forever/hist-rag-bench-private-texts"

PUBLIC_TEXTS_REPO: str = "ai-forever/hist-rag-bench-public-texts"
PUBLIC_QA_REPO: str = "ai-forever/hist-rag-bench-public-questions"


def get_private_qa_dataset(version):
    return load_dataset(HIST_PRIVATE_QA_REPO_ID, revision=version, cache_dir=CACHE_DIR)


def get_private_texts_dataset(version):
    return load_dataset(HIST_PRIVATE_TEXTS_REPO_ID, revision=version, cache_dir=CACHE_DIR)

# map public_id:private_id
def get_public_to_private_texts_mapping(version):
    private_texts_ds = get_private_texts_dataset(version)
    mapping = {}
    for item in private_texts_ds["train"]:
        mapping[item["public_id"]] = item["id"]
    return mapping


version = helper.get_latest_version(helper.get_ds_versions(PUBLIC_TEXTS_REPO))
texts_ds = load_dataset(PUBLIC_TEXTS_REPO, revision=version, cache_dir=CACHE_DIR)
questions_ds = load_dataset(PUBLIC_QA_REPO, revision=version, cache_dir=CACHE_DIR)
private_texts_ds = get_private_texts_dataset(version)
qa_dataset = get_private_qa_dataset(version)
mapping = get_public_to_private_texts_mapping(version)
version

'1.15.0'

In [4]:
COLLECTION_NAME: str = "dragon"
CHROMA_PATH: str = "../chroma_db"

chroma_client = chromadb.PersistentClient(path=CHROMA_PATH)
def create_collection(chroma_client, name):
    collection = None
    try:
        chroma_client.delete_collection(name)
    except Exception as e:
        print(f"Коллекция {name} уже существует и будет перезатерта: {e}")
    finally:
        collection = chroma_client.create_collection(
            name=name,
            metadata={
                "hnsw:space": "cosine",
            }
        )
        print(f"Коллекция {name} создана по пути {CHROMA_PATH}")
    return collection

## Вспомогательные функции

In [5]:
def get_retriever_result(query, collection, n=3):
    results = ChromaRetriever(collection).top_n(query=query, n=n)
    found_ids = [int(x["id"]) for x in results["metadatas"][0]]
    relevant_chunks = results["documents"][0]
    metadatas = results["metadatas"][0]
    return found_ids, relevant_chunks, metadatas


def evaluate_retriever(collection, qa_dataset, mapping):
    public_questions = [
        {"id": str(row["public_id"]), "question": row["question"]}
        for row in qa_dataset["train"]
    ]
    results = {}
    for q in public_questions:
        q_id = q["id"]
        found_ids, _, _ = get_retriever_result(q["question"], collection)
        results[q_id] = {"found_ids": found_ids, "model_answer": "-"}

    ev_res = evaluator.evaluate_rag_results(results, qa_dataset, mapping, evaluator.RAGEvaluator())
    return ev_res["average_metrics"]["retrieval"]

## Бенчмарк 1: сравнение сплитеров

Фиксированные параметры: `chunk_size=500`, `chunk_overlap=50`.
Сравниваем все 4 сплитера по метрикам ретривера.

In [6]:
CHUNK_SIZE = 500
CHUNK_OVERLAP = 50

benchmark1_results = {}

for splitter_name in SPLITTER_NAMES:
    print(f"\n--- {splitter_name} (chunk_size={CHUNK_SIZE}) ---")
    collection = create_collection(chroma_client, name=f"bench1_{splitter_name}")
    n_chunks = insert_data(
        dataset=texts_ds,
        collection=collection,
        split_type=splitter_name,
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP,
    )
    metrics = evaluate_retriever(collection, qa_dataset, mapping)
    benchmark1_results[splitter_name] = {"n_chunks": n_chunks, **metrics}
    print(f"Метрики: {metrics}")

Created a chunk of size 585, which is longer than the specified 500
Created a chunk of size 1194, which is longer than the specified 500
Created a chunk of size 517, which is longer than the specified 500



--- character (chunk_size=500) ---
Коллекция bench1_character уже существует и будет перезатерта: Collection [bench1_character] does not exist
Коллекция bench1_character создана по пути ../chroma_db
Добавлено 564 чанков в коллекцию 'bench1_character'
Метрики: {'hit_rate': np.float64(0.83), 'mrr': np.float64(0.7477777777777778), 'precision': np.float64(0.2866666666666667)}

--- recursive_character (chunk_size=500) ---
Коллекция bench1_recursive_character уже существует и будет перезатерта: Collection [bench1_recursive_character] does not exist
Коллекция bench1_recursive_character создана по пути ../chroma_db
Добавлено 2354 чанков в коллекцию 'bench1_recursive_character'
Метрики: {'hit_rate': np.float64(0.8783333333333333), 'mrr': np.float64(0.8225), 'precision': np.float64(0.5122222222222222)}

--- token (chunk_size=500) ---
Коллекция bench1_token уже существует и будет перезатерта: Collection [bench1_token] does not exist
Коллекция bench1_token создана по пути ../chroma_db
Добавлено 2

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 585.70it/s, Materializing param=pooler.dense.weight]                               
BertModel LOAD REPORT from: intfloat/multilingual-e5-small
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Добавлено 671 чанков в коллекцию 'bench1_sentence_transformers'
Метрики: {'hit_rate': np.float64(0.825), 'mrr': np.float64(0.7441666666666666), 'precision': np.float64(0.31777777777777777)}


### Результаты бенчмарка 1

In [7]:
df1 = pd.DataFrame(benchmark1_results).T
df1.index.name = "splitter"
df1

,n_chunks,hit_rate,mrr,precision
splitter,,,,
character,564.0,0.830000,0.747778,0.286667
recursive_character,2354.0,0.878333,0.822500,0.512222
token,2148.0,0.880000,0.814722,0.507222
sentence_transformers,671.0,0.825000,0.744167,0.317778


## Бенчмарк 2: chunk_size для RecursiveCharacterTextSplitter

Фиксированный сплитер: `recursive_character`, `chunk_overlap=50`.
Сравниваем размеры чанков: 250, 500, 1000.

In [8]:
CHUNK_SIZES = [250, 500, 1000]
SPLITTER = "recursive_character"
CHUNK_OVERLAP = 50

benchmark2_results = {}

for chunk_size in CHUNK_SIZES:
    label = f"chunk_{chunk_size}"
    print(f"\n--- chunk_size={chunk_size} ---")
    collection = create_collection(chroma_client, name=f"bench2_{label}")
    n_chunks = insert_data(
        dataset=texts_ds,
        collection=collection,
        split_type=SPLITTER,
        chunk_size=chunk_size,
        chunk_overlap=CHUNK_OVERLAP,
    )
    metrics = evaluate_retriever(collection, qa_dataset, mapping)
    benchmark2_results[label] = {"chunk_size": chunk_size, "n_chunks": n_chunks, **metrics}
    print(f"Метрики: {metrics}")


--- chunk_size=250 ---
Коллекция bench2_chunk_250 уже существует и будет перезатерта: Collection [bench2_chunk_250] does not exist
Коллекция bench2_chunk_250 создана по пути ../chroma_db
Добавлено 4861 чанков в коллекцию 'bench2_chunk_250'
Метрики: {'hit_rate': np.float64(0.8816666666666667), 'mrr': np.float64(0.8205555555555556), 'precision': np.float64(0.5577777777777777)}

--- chunk_size=500 ---
Коллекция bench2_chunk_500 уже существует и будет перезатерта: Collection [bench2_chunk_500] does not exist
Коллекция bench2_chunk_500 создана по пути ../chroma_db
Добавлено 2354 чанков в коллекцию 'bench2_chunk_500'
Метрики: {'hit_rate': np.float64(0.88), 'mrr': np.float64(0.8241666666666667), 'precision': np.float64(0.5127777777777777)}

--- chunk_size=1000 ---
Коллекция bench2_chunk_1000 уже существует и будет перезатерта: Collection [bench2_chunk_1000] does not exist
Коллекция bench2_chunk_1000 создана по пути ../chroma_db
Добавлено 1151 чанков в коллекцию 'bench2_chunk_1000'
Метрики: {

In [9]:
df2 = pd.DataFrame(benchmark2_results).T
df2.index.name = "chunk_size"
df2

,chunk_size,n_chunks,hit_rate,mrr,precision
chunk_size,,,,,
chunk_250,250.0,4861.0,0.881667,0.820556,0.557778
chunk_500,500.0,2354.0,0.880000,0.824167,0.512778
chunk_1000,1000.0,1151.0,0.840000,0.764722,0.388889


In [ ]:
with open("../results/splitter_benchmark_results.json", "w", encoding="utf-8") as f:
    json.dump({"benchmark1": benchmark1_results, "benchmark2": benchmark2_results}, f, ensure_ascii=False, indent=2)

print("Результаты сохранены: results/splitter_benchmark_results.json")